In [1]:
import pandas as pd
import time

train_df = pd.read_csv('../data/processed/train_FD001_preprocessed.csv')
sensor_cols = [c for c in train_df.columns if c.startswith('sensor_')]

print("Shape:", train_df.shape)
print("Sensors:", sensor_cols)
print("Engines:", train_df['engine_id'].nunique())

Shape: (20631, 20)
Sensors: ['sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21']
Engines: 100


In [2]:
# Melt from wide to long format
train_long = train_df.melt(
    id_vars=['engine_id', 'cycle'],
    value_vars=sensor_cols,
    var_name='sensor',
    value_name='value'
)

# tsfresh needs a combined id column (engine + sensor)
train_long['id'] = train_long['engine_id'].astype(str) + '_' + train_long['sensor']

print("Long format shape:", train_long.shape)
print("\nSample:")
print(train_long.head(6))

Long format shape: (288834, 5)

Sample:
   engine_id  cycle    sensor     value          id
0          1      1  sensor_2  0.183735  1_sensor_2
1          1      2  sensor_2  0.283133  1_sensor_2
2          1      3  sensor_2  0.343373  1_sensor_2
3          1      4  sensor_2  0.343373  1_sensor_2
4          1      5  sensor_2  0.349398  1_sensor_2
5          1      6  sensor_2  0.268072  1_sensor_2


In [6]:
from tsfresh import extract_features
from tsfresh.feature_extraction import EfficientFCParameters
import time

print("Starting feature extraction...")
start_time = time.time()

features = extract_features(
    train_long,
    column_id='id',
    column_sort='cycle',
    column_value='value',
    default_fc_parameters=EfficientFCParameters(),
    n_jobs=0,           # ← changed from -1 to 0 (single process, works on Windows)
    disable_progressbar=False
)

elapsed = time.time() - start_time
print(f"\nDone! Time taken: {elapsed:.1f} seconds")
print("Features shape:", features.shape)

Starting feature extraction...


Feature Extraction: 100%|██████████| 1400/1400 [03:21<00:00,  6.95it/s]



Done! Time taken: 202.4 seconds
Features shape: (1400, 777)


In [7]:
import numpy as np

# The index looks like "1_sensor_2" - split it back into engine_id and sensor
features_reset = features.copy()
features_reset.index = features_reset.index.astype(str)

# Extract engine_id from the index (everything before the first underscore)
features_reset['engine_id'] = features_reset.index.str.split('_').str[0].astype(int)

# Group by engine_id - average across sensors for same engine
features_per_engine = features_reset.groupby('engine_id').mean()

print("Per-engine features shape:", features_per_engine.shape)

Per-engine features shape: (100, 777)


In [8]:
# Get the final RUL per engine (RUL at last cycle = 0 for training)
# But we want max RUL (at cycle 1) as the label for whole-engine prediction
rul_labels = train_df.groupby('engine_id')['RUL'].max().reset_index()
rul_labels.columns = ['engine_id', 'RUL']

print("RUL labels shape:", rul_labels.shape)
print(rul_labels.head())

RUL labels shape: (100, 2)
   engine_id  RUL
0          1  191
1          2  286
2          3  178
3          4  188
4          5  268


In [9]:
import os

# Merge features with RUL labels
features_per_engine = features_per_engine.reset_index()
dataset = features_per_engine.merge(rul_labels, on='engine_id')

print("Final dataset shape:", dataset.shape)
print("RUL column present:", 'RUL' in dataset.columns)

# Save
os.makedirs('../data/processed', exist_ok=True)
dataset.to_csv('../data/processed/features_with_rul.csv', index=False)
print("Saved to data/processed/features_with_rul.csv")

Final dataset shape: (100, 779)
RUL column present: True
Saved to data/processed/features_with_rul.csv


In [10]:
# Save runtime for reporting (bonus criteria)
import json

runtime_log = {
    'feature_extraction_seconds': round(elapsed, 1),
    'n_engines': 100,
    'n_sensors': 14,
    'n_features_raw': features.shape[1],
    'n_features_per_engine': dataset.shape[1] - 2  # exclude engine_id and RUL
}

with open('../outputs/runtime_log.json', 'w') as f:
    json.dump(runtime_log, f, indent=2)

print("Runtime log saved:")
print(json.dumps(runtime_log, indent=2))

Runtime log saved:
{
  "feature_extraction_seconds": 202.4,
  "n_engines": 100,
  "n_sensors": 14,
  "n_features_raw": 777,
  "n_features_per_engine": 777
}


## Notebook 02 — Feature Extraction with tsfresh

- Reshaped data to long format (288,834 rows) for tsfresh
- Extracted 777 features per engine using EfficientFCParameters
- Grouped features back to per-engine format (100 rows × 777 features)
- Attached RUL labels (range: 178–361)
- Runtime: 202.4 seconds
- Saved to data/processed/features_with_rul.csv